# ಪಾಠ 05 - ಏಜೆಂಟಿಕ್ RAG


## ಸೆಟ್‌ಅಪ್

ಈ ನೋಟ್‌ಬುಕ್ ಮೈಕ್ರೋಸಾಫ್ಟ್ ಏಜೆಂಟ್ ಫ್ರೇಮ್ವರ್ಕ್ ಬಳಸಿ ಏಜೆಂಟಿಕ್ RAG (ರಿಟ್ರೀವಲ್-ಅಗ್ಮೆಂಟೆಡ್ ಜನರೇಶನ್) ಪ್ಯಾಟರ್ನ್ ಅನ್ನು ಪ್ರದರ್ಶಿಸುತ್ತದೆ.

**ಅಗತ್ಯಾಗಿರುವವು:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — ನಿಮ್ಮ ಅಜ್ಯೂರ್ AI ಶೋಧನಾ ಸೇವೆ ಎಂಡ್ಪಾಯಿಂಟ್
- `AZURE_SEARCH_API_KEY` — ನಿಮ್ಮ ಅಜ್ಯೂರ್ AI ಶೋಧನಾ API ಕೀ
- ಪರಿಸರ ಚರಗಳಲ್ಲಿ ಸಂರಚಿತ ಅಜ್ಯೂರ್ ಓಪನ್AI ನಿಯೋಜನೆ
- ಅಜ್ಯೂರ್ CLI ದೃಢೀಕರಿಸಲಾಗಿದೆ (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## ಏಜೆಂಟಿಕ್ RAG ಎಂದರೆ ಏನು?

ಪಾರಂಪರಿಕ RAG ನಿಗದಿತ ಪೈಪ್ಲೈನ್ ಅನ್ನು ಅನುಸರಿಸುತ್ತದೆ: ದಾಖಲೆಗಳನ್ನು ಹಿಂತಿರುಗಿಸಿ, ನಂತರ ಪ್ರತಿಕ್ರಿಯೆಯನ್ನು ರಚಿಸಿ. **ಏಜೆಂಟಿಕ್ RAG** ಇವನ್ನು ಮೀರಿದೆ ಏಜೆಂಟ್‌ಗೆ ಸ್ವಾಯತ್ತತೆಯನ್ನು ನೀಡಲು **ಯಾವಾಗ** ಮತ್ತು **ಹೇಳಿದಂತೆ** ಮಾಹಿತಿ ಹಿಂಪಡೆಯಬೇಕೆಂದು ನಿರ್ಧರಿಸಲು.

ಏಜೆಂಟಿಕ್ RAG ಜೊತೆಗೆ, ಏಜೆಂಟ್ ಸಾಧ್ಯವೇ:
- **ನಿರ್ಧರಿಸುವುದು** ಪ್ರಶ್ನೆಗೆ ಉತ್ತರಿಸುವ ಮೊದಲು ಹಿಂಪಡೆಯುವುದು ಅಗತ್ಯವೋ ಎಂದು
- **ಆಯ್ಕೆಮಾಡುವುದು** ಯಾವ ಡೇಟಾ ಮೂಲ ಅಥವಾ ಸಾಧನವನ್ನು ಪ್ರಶ್ನಿಸಲು
- **ಮೌಲ್ಯಮಾಪನ** ಹಿಂಪಡೆಯಲಾದ ಫಲಿತಾಂಶಗಳನ್ನು ಮತ್ತು ಮೊದಲ ಪ್ರಯತ್ನದ ವೈಫಲ್ಯನಿದ್ದರೆ ನಂತರದ ಹಿಂಪಡೆಯುವ ಕಾರ್ಯಗಳು
- **ಸಂಯೋಜನೆ** ಒಂದಕ್ಕಿಂತ ಹೆಚ್ಚು ಹಿಂಪಡೆಯುವ ಹಂತಗಳಿಂದ ಮಾಹಿತಿಯನ್ನು ಸुस الهندಿತ ಉತ್ತರವಾಗಿ

ಇದು ಏಜೆಂಟ್ ಅನ್ನು ಸ್ಥಿರ retrieve-then-generate ಪೈಪ್ಲೈನಿಗಿಂತ ಹೆಚ್ಚು ಸುಗಮ ಮತ್ತು ನಿಖರಗೊಳಿಸುತ್ತದೆ.


## ಶೋಧ ಉಪಕರಣವನ್ನು ರಚಿಸಲಾಗುತ್ತಿದೆ

Agentic RAG ನಲ್ಲಿ, ಹೊರಗಿನ ಡೇಟಾ ಮೂಲಗಳನ್ನು **ಉಪಕರಣಗಳಾಗಿ** ಮಡಿತರಿಸಲಾಗುತ್ತದೆ, ಅವುಗಳನ್ನು ಏಜೆಂಟ್ ಬೇಡಿಕೆಯ ಮೇಲೆ ಕರೆ ಮಾಡಬಹುದು. ಇದರಿಂದ ಏಜೆಂಟ್‌ಗೆ ರಿಟ್ರೀವಲ್ ಅನ್ನು ಕೇವಲ ಇನ್ನೊಂದು ಕ್ರಿಯೆಯಾಗಿ ಕಾಣಬಹುದು, ಬಾಧ್ಯತೆಯಾದ ಹಂತವಲ್ಲದೆ.

ಕೆಳಗಿನವುಗಳಲ್ಲಿ ನಾವು ಪ್ರವಾಸ ಮಾಹಿತಿ ಭಂಡಾರವನ್ನು ವ್ಯಾಖ್ಯಾನಿಸಿ, ಅದನ್ನು ಏಜೆಂಟ್ ಗುರಿ ತಾಣದ ಮಾಹಿತಿಯನ್ನು ಹುಡುಕಲು ಕರೆ ಮಾಡಬಹುದಾದ ಉಪಕರಣವಾಗಿ ಪ್ರದರ್ಶಿಸುತ್ತೇವೆ.


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## RAG ಏಜೆಂಟ್ ನಿರ್ಮಾಣ

ಈಗ ನಾವು ಆಜ್ಞಾಪಿಸಲಾದ ಏಜೆಂಟ್ ಅನ್ನು ರಚಿಸುತ್ತೇವೆ, ಅದು **ಪ್ರತಿಕ್ರಿಯಿಸುವ ಮೊದಲು ಯಾವಾಗಲೂ ಮಾಹಿತಿ ಪಡೆಯಬೇಕು**. ಈ ಏಜೆಂಟ್ ತನ್ನ ಉತ್ತರಗಳನ್ನು ತನ್ನ ಸ್ವಂತ ತರಬೇತಿ ಡೇಟಾದ ಮೇಲೆ ಅವಲಂಬಿಸುವ 대신 ಜ್ಞಾನ ಆಧಾರದ ಮೇಲೆ ಸ್ಥಾಪಿಸಲು `search_travel_knowledge` ಸಾಧನವನ್ನು ಬಳಸುತ್ತದೆ.


In [ ]:
agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## ಪುನರಾವರ್ತಿತ ಪಡೆಯುವಿಕೆ — ತಯಾರಕರು-ಪರಿಶೀಲಕರು ಮಾದರಿ

Agentic RAG ನ ಪ್ರಮುಖ ಲಾಭ **ಪುನರಾವರ್ತಿತ ಪಡೆಯುವಿಕೆ**. ಏಜೆಂಟ್ ತನ್ನ ಹೊಣೆಯ ಹಿಡಿದ ಮಾಹಿತಿಗಳನ್ನ ಪರಿಶೀಲಿಸಲು, ಸುಧಾರಿಸಲು, ಅಥವಾ ವಿಸ್ತರಿಸಲು ಹಲವು ಸುತ್ತುಗಳ ಸರ್ಚ್ ನಡೆಸಬಹುದು — ಇದು "ತಯಾರಕರು-ಪರಿಶೀಲಕರು" ಕೆಲಸದ ತಂತ್ರದಂತೆ:

1. **ತಯಾರಕರು ಹಂತ**: ಏಜೆಂಟ್ ಪ್ರಾಥಮಿಕ ಮಾಹಿತಿಯನ್ನು ತೆಗೆದುಕೊಂಡು ಉತ್ತರದ ಒದಗಿಸುವಿಕೆಯನ್ನು ರೂಪಿಸುತ್ತದೆ.
2. **ಪರಿಶೀಲಕರು ಹಂತ**: ಏಜೆಂಟ್ ವಿವರಗಳನ್ನು ಪರಿಶೀಲಿಸಲು ಅಥವಾ ಖಾಲಿ ತೂಗುಗಳನ್ನು ತುಂಬಲು ಹೆಚ್ಚುವರಿ ಪಡೆಯುವಿಕೆಗಳನ್ನು ನಡೆಸುತ್ತದೆ.

ಕೆಳಗೆ, ಏಜೆಂಟ್‌ಗೊಬ್ಬರಿಗೆ ಹೋಲಿಸುವбель ಪ್ರಾಪ್ತಿಸುವ ಸ್ಥಳಗಳನ್ನು ಹೋಲಿಸಲು ಪ್ರಶ್ನೆ ಕೇಳಲಾಗಿದೆ, ಇದರಿಂದ ಅದನ್ನು ಹಲವು ಬಾರಿ ಹುಡುಕಲು ನಿರ್ಬಂಧಿಸಲಾಗುತ್ತದೆ.


In [ ]:
checker_agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## Summary

ಈ ಪಾಠದಲ್ಲಿ ನೀವು Microsoft Agent Framework ಬಳಸಿ **Agentic RAG** ಸಿಸ್ಟಂ ಅನ್ನು ಹೇಗೆ ನಿರ್ಮಿಸಬೇಕೆಂಬುದನ್ನು ಕಲಿದಿರಿ:

- **Agentic RAG** ಏಜೆಂಟ್‌ಗಳಿಗೆ ಸ್ವತಂತ್ರವಾಗಿ ಮಾಹಿತಿಯನ್ನು ಎപ്പോൾ ಪಡೆಯಬೇಕೆಂದು ತೀರ್ಮಾನಿಸುವ ಅವಕಾಶವನ್ನು ನೀಡುತ್ತದೆ, ಇದರಿಂದ retrieval ನಿಗದಿತವಲ್ಲದೆ ಸಕ್ರಮವಾಗುತ್ತದೆ.
- **ಟೂಲ್ಗಳು ಡೇಟಾ ಮೂಲಗಳಾಗಿ**: ಹೊರಗಿನ ಜ್ಞಾನ ನಿಲ್ದಾಣಗಳು (ಉದಾಹರಣೆಗೆ Azure AI Search) ಏಜೆಂಟ್ ಬಳಸಬಹುದಾದ ಟೂಲ್‌ಗಳಾಗಿ ಮೊಡಕಿಸಲಾಗುತ್ತವೆ.
- **ಪುನರಾವೃತಾದ retrieval**: maker-checker ಮಾದರಿ ಏಜೆಂಟ್‌ಗೆ ಒಂದು ಅಂತಿಮ ಉತ್ತರವನ್ನು ರಚಿಸಲು ಹಲವು retrieval ಸುತ್ತುಗಳನ್ನು — ಹುಡುಕಾಟ, ಪರಿಶೀಲನೆ ಮತ್ತು ಪರಿಶುದ್ಧೀಕರಣ — ನೆರವೇರಿಸುವ ಅವಕಾಶ ನೀಡುತ್ತದೆ.

ಉತ್ಪಾದನೆಯಲ್ಲಿ, ನೀವು ಸ್ಮೃತಿ ಆಧಾರಿತ `TRAVEL_KNOWLEDGE_BASE` ಅನ್ನು ವಿಸ್ತೃತ ಪ್ರಯಾಣ ಡಾಕ್ಯುಮೆಂಟ್ retrieval ನಿಂದ ನಿಭಾಯಿಸಲು ನಿಜವಾದ Azure AI Search ಸೂಚ್ಯಂಕದೊಂದಿಗೆ ಬದಲಾಯಿಸುತ್ತೀರಿ.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ಅಸ್ವೀಕಾರ**:
ಈ ದಸ್ತಾವೇಜು AI ಅನುವಾದ ಸೇವೆ [Co-op Translator](https://github.com/Azure/co-op-translator) ಬಳಸಿ ಅನುವಾದಿಸಲಾಗಿದೆ. ನಾವು ನಿಖರತೆಯನ್ನು ಸಾಧಿಸಲು ಪ್ರಯತ್ನಿಸುತ್ತಿದ್ದರೂ, ದಯವಿಟ್ಟು ಗಮನಿಸಿ, ಸ್ವಯಂಚಾಲಿತ ಅನುವಾದಗಳಲ್ಲಿ ದೋಷಗಳು ಅಥವಾ ಅಸಡ್ಡೆಗಳು ಇರಬಹುದು. ಮೂಲ ಭಾಷೆಯಲ್ಲಿರುವ ಮೂಲ ದಸ್ತಾವೇಜು ಪ್ರಾಮಾಣಿಕ ಮೂಲವೆಂದು ಪರಿಗಣಿಸಬೇಕು. ಪ್ರಮುಖ ಮಾಹಿತಿಗಾಗಿ, ವೃತ್ತಿಪರ ಮಾನವ ಅನುವಾದವನ್ನು ಶಿಫಾರಸು ಮಾಡಲಾಗುತ್ತದೆ. ಈ ಅನುವಾದವನ್ನು ಬಳಸುವ ಮೂಲಕ ಉಂಟಾಗುವ ಯಾವುದೇ ತಪ್ಪು ಅರ್ಥಗಳ ಅಥವಾ ತಪ್ಪು ವ್ಯಾಖ್ಯಾನಗಳ ಬಗ್ಗೆ ನಾವು ಹೊಣೆಗಾರರಲ್ಲ.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
